In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q albumentations==1.3.1 timm grad-cam

In [ ]:
import os, cv2, copy, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    accuracy_score, classification_report,
    roc_curve, auc, precision_recall_curve,
    average_precision_score, confusion_matrix
)

BASE_DIR  = '/kaggle/input/datasets/andrewmvd/retinal-disease-classification'
TRAIN_IMG = os.path.join(BASE_DIR, 'Training_Set/Training_Set/Training')
VAL_IMG   = os.path.join(BASE_DIR, 'Evaluation_Set/Evaluation_Set/Validation')
TEST_IMG  = os.path.join(BASE_DIR, 'Test_Set/Test_Set/Test')
TRAIN_CSV = os.path.join(BASE_DIR, 'Training_Set/Training_Set/RFMiD_Training_Labels.csv')
VAL_CSV   = os.path.join(BASE_DIR, 'Evaluation_Set/Evaluation_Set/RFMiD_Validation_Labels.csv')
TEST_CSV  = os.path.join(BASE_DIR, 'Test_Set/Test_Set/RFMiD_Testing_Labels.csv')

DISEASE_COLS = ['DR', 'ARMD', 'MH', 'DN', 'MYA', 'TSLN']
IMAGE_SIZE   = 224
BATCH_SIZE   = 16
NUM_EPOCHS   = 40
LR           = 1e-4
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Device:', DEVICE)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
def apply_clahe(image):
    lab   = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    lab[:,:,0] = clahe.apply(lab[:,:,0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def get_train_transforms():
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.4),
        A.GaussNoise(var_limit=(10.0,50.0), p=0.2),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2()
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2()
    ])

print('Transforms ready')

In [ ]:
def load_rfmid_df(csv_path, img_dir):
    df = pd.read_csv(csv_path)
    df = df[['ID'] + DISEASE_COLS].copy()
    df['image_path'] = df['ID'].apply(lambda x: os.path.join(img_dir, f'{x}.png'))
    return df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)

train_df = load_rfmid_df(TRAIN_CSV, TRAIN_IMG)
val_df   = load_rfmid_df(VAL_CSV,   VAL_IMG)
test_df  = load_rfmid_df(TEST_CSV,  TEST_IMG)

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
print('\nPositive counts (train):')
print(train_df[DISEASE_COLS].sum())

class RFMiDDataset(Dataset):
    def __init__(self, dataframe, transforms, use_clahe=True):
        self.df = dataframe.reset_index(drop=True)
        self.transforms = transforms
        self.use_clahe  = use_clahe
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = cv2.imread(row['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.use_clahe: image = apply_clahe(image)
        image = self.transforms(image=image)['image']
        label = torch.tensor(row[DISEASE_COLS].values.astype(np.float32))
        return image, label

In [ ]:
# ── Imbalance handling ────────────────────────────────────────
pos_counts   = train_df[DISEASE_COLS].sum().values
neg_counts   = len(train_df) - pos_counts
pos_weight   = torch.tensor(neg_counts / pos_counts, dtype=torch.float32).to(DEVICE)

print('pos_weight per disease:')
for n, w in zip(DISEASE_COLS, pos_weight.cpu()): print(f'  {n:<6}: {w:.2f}')

label_weights  = pos_weight.cpu().numpy()
sample_weights = []
for _, row in train_df.iterrows():
    lbl = row[DISEASE_COLS].values.astype(np.float32)
    w   = float(np.max(lbl * label_weights)) if lbl.sum() > 0 else 1.0
    sample_weights.append(w)

sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
print(f'Sampler ready — {len(sample_weights)} sample weights assigned')

In [ ]:
train_ds = RFMiDDataset(train_df, get_train_transforms())
val_ds   = RFMiDDataset(val_df,   get_val_transforms())
test_ds  = RFMiDDataset(test_df,  get_val_transforms())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,      num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,         num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,         num_workers=2, pin_memory=True)

images, labels = next(iter(train_loader))
print(f'Image batch: {images.shape}  Label batch: {labels.shape}')

In [ ]:
class HybridRetinalModel(nn.Module):
    def __init__(self, num_classes=6, dropout=0.4):
        super().__init__()
        self.cnn = timm.create_model('efficientnet_b4', pretrained=True, num_classes=0)
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        cnn_dim, vit_dim, FUSION_DIM = self.cnn.num_features, self.vit.num_features, 512

        self.cnn_proj = nn.Sequential(nn.Linear(cnn_dim, FUSION_DIM), nn.LayerNorm(FUSION_DIM), nn.GELU())
        self.vit_proj = nn.Sequential(nn.Linear(vit_dim, FUSION_DIM), nn.LayerNorm(FUSION_DIM), nn.GELU())
        self.cross_attn = nn.MultiheadAttention(embed_dim=FUSION_DIM, num_heads=8, dropout=0.1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(FUSION_DIM * 2, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(256, num_classes))

    def forward(self, x):
        cnn_feat = self.cnn_proj(self.cnn(x))
        vit_feat = self.vit_proj(self.vit(x))
        attended, _ = self.cross_attn(cnn_feat.unsqueeze(1), vit_feat.unsqueeze(1), vit_feat.unsqueeze(1))
        fused = torch.cat([cnn_feat, attended.squeeze(1)], dim=1)
        return self.classifier(fused)

model = HybridRetinalModel(num_classes=6, dropout=0.4).to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f'Total params: {total:,}')
with torch.no_grad():
    out = model(torch.randn(2,3,224,224).to(DEVICE))
    print(f'Output shape: {out.shape}')

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
print('Loss / Optimizer / Scheduler ready')

In [ ]:
def compute_accuracy(labels, probs, threshold=0.5):
    preds     = (probs >= threshold).astype(int)
    exact     = (preds == labels).all(axis=1).mean()
    per_label = (preds == labels).mean(axis=0)
    return exact, per_label

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, all_labels, all_probs = 0, [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())
    all_probs  = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    aucs = []
    for i in range(len(DISEASE_COLS)):
        try:    aucs.append(roc_auc_score(all_labels[:,i], all_probs[:,i]))
        except: aucs.append(0.0)
    exact_acc, _ = compute_accuracy(all_labels, all_probs)
    return total_loss/len(loader), np.mean(aucs), aucs, float(exact_acc)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_labels, all_probs = 0, [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        total_loss += criterion(logits, labels).item()
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    all_probs  = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    aucs = []
    for i in range(len(DISEASE_COLS)):
        try:    aucs.append(roc_auc_score(all_labels[:,i], all_probs[:,i]))
        except: aucs.append(0.0)
    exact_acc, per_label_acc = compute_accuracy(all_labels, all_probs)
    return total_loss/len(loader), np.mean(aucs), aucs, float(exact_acc), per_label_acc, all_labels, all_probs

print('Train / eval functions ready')

In [ ]:
history = {'train_loss':[],'val_loss':[],'train_auc':[],'val_auc':[],'train_acc':[],'val_acc':[]}

best_val_auc, no_improve, PATIENCE = 0.0, 0, 5
best_model_wts = copy.deepcopy(model.state_dict())

print(f"{'Ep':>3} | {'TrLoss':>7} | {'VlLoss':>7} | {'TrAUC':>6} | {'VlAUC':>6} | {'TrAcc':>6} | {'VlAcc':>6} | Per-class Val AUC")
print('-'*115)

for epoch in range(1, NUM_EPOCHS+1):

    # 🔹 Training
    tr_loss, tr_auc, _, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)

    # 🔹 Validation
    vl_loss, vl_auc, vl_aucs, vl_acc, _, _, _ = evaluate(model, val_loader, criterion, DEVICE)

    # 🔥 FIX 1: Use scheduler correctly (only if ReduceLROnPlateau)
    scheduler.step(vl_auc)

    # 🔹 Save history
    for k,v in zip(['train_loss','val_loss','train_auc','val_auc','train_acc','val_acc'],
                   [tr_loss, vl_loss, tr_auc, vl_auc, tr_acc, vl_acc]):
        history[k].append(v)

    # 🔥 FIX 2: Allow small improvements
    if vl_auc > best_val_auc + 1e-4:
        best_val_auc = vl_auc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), 'best_model.pth')
        tag, no_improve = ' best', 0
    else:
        tag = ''
        no_improve += 1

    # 🔹 Print results
    per_cls = '  '.join([f'{n}:{v:.3f}' for n,v in zip(DISEASE_COLS, vl_aucs)])
    print(f"{epoch:>3} | {tr_loss:>7.4f} | {vl_loss:>7.4f} | {tr_auc:>6.4f} | {vl_auc:>6.4f} | {tr_acc:>6.4f} | {vl_acc:>6.4f} | {per_cls}{tag}")

    # 🔹 Early stopping
    if no_improve >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

print(f'\nBest Val AUC: {best_val_auc:.4f}')
model.load_state_dict(best_model_wts)

In [ ]:
# ── Training curves: Loss + AUC + Accuracy ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(18,5))
epochs_ran = range(1, len(history['train_loss'])+1)

for ax, tr_key, vl_key, title, ylabel in [
    (axes[0], 'train_loss', 'val_loss', 'Loss Curve',              'Loss'),
    (axes[1], 'train_auc',  'val_auc',  'Mean AUC Curve',          'AUC'),
    (axes[2], 'train_acc',  'val_acc',  'Exact-match Accuracy',    'Accuracy')]:
    ax.plot(epochs_ran, history[tr_key], label='Train', color='steelblue', linewidth=2)
    ax.plot(epochs_ran, history[vl_key], label='Val',   color='coral',     linewidth=2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Optimal thresholds via Youden's J on validation ──────────
from sklearn.metrics import roc_curve

_, _, _, _, _, val_labels, val_probs = evaluate(model, val_loader, criterion, DEVICE)

optimal_thresholds = {}
print('Optimal thresholds (Youden J):')
print('-'*30)
for i, name in enumerate(DISEASE_COLS):
    fpr, tpr, threshs = roc_curve(val_labels[:,i], val_probs[:,i])
    best_t = float(np.clip(threshs[np.argmax(tpr - fpr)], 0.1, 0.9))
    optimal_thresholds[name] = best_t
    print(f'  {name:<6}: {best_t:.3f}')

In [ ]:
# ── Full test set evaluation ──────────────────────────────────
_, _, _, _, _, test_labels, test_probs = evaluate(model, test_loader, criterion, DEVICE)

test_preds = np.zeros_like(test_probs, dtype=int)
for i, name in enumerate(DISEASE_COLS):
    test_preds[:,i] = (test_probs[:,i] >= optimal_thresholds[name]).astype(int)

print(f'\n{"Disease":<8} {"AUC":>6}  {"F1":>6}  {"Prec":>6}  {"Rec":>6}  {"Acc":>6}')
print('='*48)
results = {}
for i, name in enumerate(DISEASE_COLS):
    a  = roc_auc_score(test_labels[:,i], test_probs[:,i])
    f  = f1_score(test_labels[:,i], test_preds[:,i], zero_division=0)
    p  = precision_score(test_labels[:,i], test_preds[:,i], zero_division=0)
    r  = recall_score(test_labels[:,i], test_preds[:,i], zero_division=0)
    ac = accuracy_score(test_labels[:,i], test_preds[:,i])
    results[name] = dict(auc=a, f1=f, precision=p, recall=r, accuracy=ac)
    print(f'{name:<8} {a:>6.4f}  {f:>6.4f}  {p:>6.4f}  {r:>6.4f}  {ac:>6.4f}')
print('='*48)
mean_auc  = roc_auc_score(test_labels, test_probs, average='macro')
mean_f1   = f1_score(test_labels, test_preds, average='macro', zero_division=0)
mean_prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
mean_rec  = recall_score(test_labels, test_preds, average='macro', zero_division=0)
exact_acc = (test_preds == test_labels).all(axis=1).mean()
print(f'{"Mean":<8} {mean_auc:>6.4f}  {mean_f1:>6.4f}  {mean_prec:>6.4f}  {mean_rec:>6.4f}  {exact_acc:>6.4f}')

In [ ]:
# ── Per-disease ROC curves ────────────────────────────────────
colors = ['#2196F3','#E91E63','#4CAF50','#FF9800','#9C27B0','#00BCD4']
fig, axes = plt.subplots(2, 3, figsize=(15,10))

for i, (name, ax) in enumerate(zip(DISEASE_COLS, axes.flatten())):
    fpr, tpr, _ = roc_curve(test_labels[:,i], test_probs[:,i])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], linewidth=2.5, label=f'AUC = {roc_auc_val:.4f}')
    ax.plot([0,1],[0,1],'k--',linewidth=1,alpha=0.5)
    ax.fill_between(fpr, tpr, alpha=0.08, color=colors[i])
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right', fontsize=11); ax.grid(alpha=0.3)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])

plt.suptitle('ROC Curves — Per Disease (Test Set)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrix per disease ──────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15,10))

for i, (name, ax) in enumerate(zip(DISEASE_COLS, axes.flatten())):
    cm      = confusion_matrix(test_labels[:,i], test_preds[:,i])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xticks([0,1]); ax.set_xticklabels(['Negative','Positive'])
    ax.set_yticks([0,1]); ax.set_yticklabels(['Negative','Positive'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    for r,c in itertools.product(range(2),range(2)):
        ax.text(c, r, f'{cm[r,c]}\n({cm_norm[r,c]:.1%})',
                ha='center', va='center', fontsize=11, fontweight='bold',
                color='white' if cm_norm[r,c]>0.5 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Confusion Matrices — Per Disease (Test Set)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

cam      = GradCAM(model=model, target_layers=[model.cnn.conv_head])
mean_np  = np.array([0.485, 0.456, 0.406])
std_np   = np.array([0.229, 0.224, 0.225])
sample_images, sample_labels = next(iter(test_loader))
n_show   = min(6, len(sample_images))

fig, axes = plt.subplots(n_show, 2, figsize=(10, n_show * 3.5))
for i in range(n_show):
    inp  = sample_images[i].unsqueeze(0).to(DEVICE)
    mask = cam(input_tensor=inp)
    img_display = np.clip(
        sample_images[i].permute(1,2,0).numpy() * std_np + mean_np,
        0, 1).astype(np.float32)
    cam_image = show_cam_on_image(img_display, mask[0], use_rgb=True)
    lbl      = sample_labels[i].numpy().astype(int)
    diseases = [DISEASE_COLS[j] for j, v in enumerate(lbl) if v]
    title    = ', '.join(diseases) if diseases else 'Normal'
    axes[i,0].imshow(img_display)
    axes[i,0].set_title(f'GT: {title}', fontsize=9)
    axes[i,0].axis('off')
    axes[i,1].imshow(cam_image)
    axes[i,1].set_title('Grad-CAM', fontsize=9)
    axes[i,1].axis('off')
plt.suptitle('Grad-CAM Explainability', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gradcam_v3.png', dpi=150, bbox_inches='tight')
plt.show()